# Snowflake Data Masking & Row Access Policies Tutorial

This notebook demonstrates **Dynamic Data Masking** and **Row Access Policies** for column-level and row-level security.

**What you'll learn:**
1. Create masking policies to hide sensitive column data based on role
2. Create row access policies to filter rows based on role
3. Apply and manage policies on tables
4. Test policies with different roles

**Pattern:** Setup → Demo → Cleanup

**Note:** Requires Enterprise Edition or higher.

## 1. Setup — Create Database, Schema, Roles & Tables

In [ ]:
%%sql -r setup_db
CREATE OR REPLACE DATABASE masking_tutorial_db;
CREATE SCHEMA masking_tutorial_db.tutorial;

In [ ]:
%%sql -r roles_result
CREATE OR REPLACE ROLE masking_admin_role;
CREATE OR REPLACE ROLE masking_analyst_role;
CREATE OR REPLACE ROLE masking_viewer_role;

GRANT ROLE masking_admin_role TO ROLE SYSADMIN;
GRANT ROLE masking_analyst_role TO ROLE SYSADMIN;
GRANT ROLE masking_viewer_role TO ROLE SYSADMIN;

In [ ]:
%%sql -r grants_result
GRANT USAGE ON DATABASE masking_tutorial_db TO ROLE masking_admin_role;
GRANT USAGE ON DATABASE masking_tutorial_db TO ROLE masking_analyst_role;
GRANT USAGE ON DATABASE masking_tutorial_db TO ROLE masking_viewer_role;
GRANT USAGE ON SCHEMA masking_tutorial_db.tutorial TO ROLE masking_admin_role;
GRANT USAGE ON SCHEMA masking_tutorial_db.tutorial TO ROLE masking_analyst_role;
GRANT USAGE ON SCHEMA masking_tutorial_db.tutorial TO ROLE masking_viewer_role;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE masking_admin_role;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE masking_analyst_role;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE masking_viewer_role;

In [ ]:
%%sql -r table_result
CREATE OR REPLACE TABLE masking_tutorial_db.tutorial.customers (
    customer_id INT,
    full_name VARCHAR(100),
    email VARCHAR(200),
    phone VARCHAR(20),
    national_insurance_no VARCHAR(13),
    salary DECIMAL(10,2),
    department VARCHAR(50),
    region VARCHAR(50)
);

## 2. Insert Sample Data

In [ ]:
%%sql -r insert_result
INSERT INTO masking_tutorial_db.tutorial.customers VALUES
    (1, 'Alice Johnson', 'alice@company.com', '+44 7700 900001', 'AB 12 34 56 C', 95000.00, 'Engineering', 'EMEA'),
    (2, 'Bob Smith', 'bob@company.com', '+44 7700 900002', 'CD 34 56 78 D', 72000.00, 'Marketing', 'EMEA'),
    (3, 'Charlie Brown', 'charlie@company.com', '+44 7700 900003', 'EF 56 78 90 E', 88000.00, 'Engineering', 'APAC'),
    (4, 'Diana Ross', 'diana@company.com', '+44 7700 900004', 'GH 78 90 12 F', 91000.00, 'Finance', 'AMER'),
    (5, 'Edward King', 'edward@company.com', '+44 7700 900005', 'IJ 90 12 34 G', 68000.00, 'Marketing', 'APAC');

In [ ]:
%%sql -r table_grants
GRANT SELECT ON TABLE masking_tutorial_db.tutorial.customers TO ROLE masking_admin_role;
GRANT SELECT ON TABLE masking_tutorial_db.tutorial.customers TO ROLE masking_analyst_role;
GRANT SELECT ON TABLE masking_tutorial_db.tutorial.customers TO ROLE masking_viewer_role;

In [ ]:
%%sql -r raw_data
SELECT * FROM masking_tutorial_db.tutorial.customers;

## 3. Create Dynamic Data Masking Policies

Masking policies define how column values appear based on the querying user's role.

In [ ]:
%%sql -r email_policy
CREATE OR REPLACE MASKING POLICY masking_tutorial_db.tutorial.email_mask
    AS (val VARCHAR)
    RETURNS VARCHAR ->
    CASE
        WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'MASKING_ADMIN_ROLE') THEN val
        WHEN CURRENT_ROLE() = 'MASKING_ANALYST_ROLE' THEN REGEXP_REPLACE(val, '.+@', '****@')
        ELSE '********'
    END;

In [ ]:
%%sql -r phone_policy
CREATE OR REPLACE MASKING POLICY masking_tutorial_db.tutorial.phone_mask
    AS (val VARCHAR)
    RETURNS VARCHAR ->
    CASE
        WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'MASKING_ADMIN_ROLE') THEN val
        ELSE CONCAT(LEFT(val, 7), ' *** ****')
    END;

In [ ]:
%%sql -r nino_policy
CREATE OR REPLACE MASKING POLICY masking_tutorial_db.tutorial.nino_mask
    AS (val VARCHAR)
    RETURNS VARCHAR ->
    CASE
        WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'MASKING_ADMIN_ROLE') THEN val
        ELSE '** ** ** ** *'
    END;

In [ ]:
%%sql -r salary_policy
CREATE OR REPLACE MASKING POLICY masking_tutorial_db.tutorial.salary_mask
    AS (val DECIMAL(10,2))
    RETURNS DECIMAL(10,2) ->
    CASE
        WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'MASKING_ADMIN_ROLE') THEN val
        WHEN CURRENT_ROLE() = 'MASKING_ANALYST_ROLE' THEN val
        ELSE NULL
    END;

## 4. Apply Masking Policies to Columns

In [ ]:
%%sql -r apply_policies
ALTER TABLE masking_tutorial_db.tutorial.customers
    MODIFY COLUMN email SET MASKING POLICY masking_tutorial_db.tutorial.email_mask;
ALTER TABLE masking_tutorial_db.tutorial.customers
    MODIFY COLUMN phone SET MASKING POLICY masking_tutorial_db.tutorial.phone_mask;
ALTER TABLE masking_tutorial_db.tutorial.customers
    MODIFY COLUMN national_insurance_no SET MASKING POLICY masking_tutorial_db.tutorial.nino_mask;
ALTER TABLE masking_tutorial_db.tutorial.customers
    MODIFY COLUMN salary SET MASKING POLICY masking_tutorial_db.tutorial.salary_mask;

## 5. Test Masking Policies with Different Roles

In [ ]:
%%sql -r accountadmin_view
SELECT
    customer_id,
    full_name,
    email,
    phone,
    national_insurance_no,
    salary
FROM masking_tutorial_db.tutorial.customers
ORDER BY customer_id;

In [ ]:
%%sql -r analyst_view
USE ROLE masking_analyst_role;
SELECT
    customer_id,
    full_name,
    email,
    phone,
    national_insurance_no,
    salary
FROM masking_tutorial_db.tutorial.customers
ORDER BY customer_id;

In [ ]:
%%sql -r viewer_view
USE ROLE masking_viewer_role;
SELECT
    customer_id,
    full_name,
    email,
    phone,
    national_insurance_no,
    salary
FROM masking_tutorial_db.tutorial.customers
ORDER BY customer_id;

In [ ]:
%%sql -r reset_role
USE ROLE ACCOUNTADMIN;

## 6. Create a Row Access Policy

Row access policies filter which rows a role can see.

In [ ]:
%%sql -r mapping_table
CREATE OR REPLACE TABLE masking_tutorial_db.tutorial.role_region_mapping (
    role_name VARCHAR(50),
    region VARCHAR(50)
);

INSERT INTO masking_tutorial_db.tutorial.role_region_mapping VALUES
    ('MASKING_ADMIN_ROLE', 'EMEA'),
    ('MASKING_ADMIN_ROLE', 'APAC'),
    ('MASKING_ADMIN_ROLE', 'AMER'),
    ('MASKING_ANALYST_ROLE', 'EMEA'),
    ('MASKING_ANALYST_ROLE', 'APAC'),
    ('MASKING_VIEWER_ROLE', 'EMEA');

GRANT SELECT ON TABLE masking_tutorial_db.tutorial.role_region_mapping TO ROLE masking_admin_role;
GRANT SELECT ON TABLE masking_tutorial_db.tutorial.role_region_mapping TO ROLE masking_analyst_role;
GRANT SELECT ON TABLE masking_tutorial_db.tutorial.role_region_mapping TO ROLE masking_viewer_role;

In [ ]:
%%sql -r rap_result
CREATE OR REPLACE ROW ACCESS POLICY masking_tutorial_db.tutorial.region_access_policy
    AS (region_val VARCHAR)
    RETURNS BOOLEAN ->
    CURRENT_ROLE() = 'ACCOUNTADMIN'
    OR EXISTS (
        SELECT 1
        FROM masking_tutorial_db.tutorial.role_region_mapping
        WHERE role_name = CURRENT_ROLE()
          AND region = region_val
    );

In [ ]:
%%sql -r apply_rap
ALTER TABLE masking_tutorial_db.tutorial.customers
    ADD ROW ACCESS POLICY masking_tutorial_db.tutorial.region_access_policy
    ON (region);

## 7. Test Row Access Policy

In [ ]:
%%sql -r admin_rows
SELECT customer_id, full_name, region
FROM masking_tutorial_db.tutorial.customers
ORDER BY customer_id;

In [ ]:
%%sql -r analyst_rows
USE ROLE masking_analyst_role;
SELECT customer_id, full_name, region
FROM masking_tutorial_db.tutorial.customers
ORDER BY customer_id;

In [ ]:
%%sql -r viewer_rows
USE ROLE masking_viewer_role;
SELECT customer_id, full_name, region
FROM masking_tutorial_db.tutorial.customers
ORDER BY customer_id;

In [ ]:
%%sql -r final_reset
USE ROLE ACCOUNTADMIN;

## 8. Audit Policies

In [ ]:
%%sql -r policy_audit
SELECT *
FROM TABLE(masking_tutorial_db.information_schema.policy_references(
    REF_ENTITY_NAME => 'masking_tutorial_db.tutorial.customers',
    REF_ENTITY_DOMAIN => 'TABLE'
));

## 9. Cleanup — Drop All Tutorial Objects

In [ ]:
%%sql -r cleanup_result
DROP DATABASE IF EXISTS masking_tutorial_db;
DROP ROLE IF EXISTS masking_admin_role;
DROP ROLE IF EXISTS masking_analyst_role;
DROP ROLE IF EXISTS masking_viewer_role;